In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool

import subprocess

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import zscore

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.columns = ['chrom', 'start', 'end']
pmds.shape

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.header.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t')
peaks.rename(columns={'seqnames' : 'chrom'}, inplace=True)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks['chrom'], peaks['start'])]
peaks.shape

In [ ]:
adata_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/h5ad/atac/peaks/clusters/justin_chromhmm_cluster_donor.h5ad'

adata = sc.read_h5ad(adata_path)
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
adata

In [ ]:
    
# pseudo_df.shape

In [ ]:
pseudo_df = pd.DataFrame(adata.X, index=adata.obs['cluster_donor'], columns=adata.var.index)
pseudo_df.shape

In [ ]:
pseudo_df.head()

In [ ]:
denom = pseudo_df.sum(axis=1)
norm_pseudo = pseudo_df.div(denom, axis=0) * 1e6
norm_pseudo.columns = adata.var.index
norm_pseudo.shape

In [ ]:
log_pseudo = np.log(norm_pseudo + 1)
log_pseudo.shape

### overlap DE peaks with PMDs

In [ ]:
s = 'Start'
tmp_peaks = peaks[peaks['coarse_v7'] == s]
tmp_peaks = tmp_peaks[~tmp_peaks.index.duplicated()]
tmp_peaks.shape

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
tmp_peaks_bt = BedTool.from_dataframe(tmp_peaks)

tmp_peaks_pmds = tmp_peaks_bt.intersect(pmds_bt, wa=True).to_dataframe()
tmp_peaks_pmds.columns = tmp_peaks.columns
tmp_peaks_pmds.index = [f'{x}_{y}' for x,y in zip(tmp_peaks_pmds['chrom'], tmp_peaks_pmds['start'])]
tmp_peaks_pmds.shape

In [ ]:
mask = ~tmp_peaks.index.isin(tmp_peaks_pmds.index)
tmp_peaks_comp = tmp_peaks[mask]
tmp_peaks_comp.shape

In [ ]:
tmp_peaks_pmds.head()

In [ ]:
s = 'Start'
mask = [x.split('_')[0] == s for x in norm_pseudo.index]
tmp_pseudo = norm_pseudo[mask]

mask = tmp_pseudo.columns.isin(tmp_peaks.index)
tmp_pseudo = tmp_pseudo.loc[:,mask]
tmp_pseudo.shape

In [ ]:
tmp_peaks

In [ ]:
merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
merged_expr.shape

In [ ]:
mask = merged_expr.index.isin(tmp_peaks_pmds.index)
merged_pmds = merged_expr[mask].copy()
merged_pmds.shape

In [ ]:
mask = ~merged_expr.index.isin(tmp_peaks_pmds.index)
merged_comp = merged_expr[mask].copy()
merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
mask = to_plot['expr'] != 0
to_plot = to_plot[mask]
to_plot.shape

In [ ]:
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'type'
y_lab = 'expr'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Log CPM', fontsize=16)
ax.set_title(f'{s} - ATAC peak expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

### all clusters now

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_peaks = peaks[peaks['coarse_v7'] == s]
    tmp_peaks = tmp_peaks[~tmp_peaks.index.duplicated()]
    tmp_peaks_bt = BedTool.from_dataframe(tmp_peaks)
    tmp_peaks_pmds = tmp_peaks_bt.intersect(pmds_bt, wa=True).to_dataframe()
    tmp_peaks_pmds.columns = tmp_peaks.columns
    tmp_peaks_pmds.index = [f'{x}_{y}' for x,y in zip(tmp_peaks_pmds['chrom'], tmp_peaks_pmds['start'])]
    
    ###
    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]
    mask = tmp_pseudo.columns.isin(tmp_peaks.index)
    tmp_pseudo = tmp_pseudo.loc[:,mask]
    
    ###
    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(tmp_peaks_pmds.index)
    merged_pmds = merged_expr[mask].copy()
    mask = ~merged_expr.index.isin(tmp_peaks_pmds.index)
    merged_comp = merged_expr[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']
    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
pmds_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'cluster'
y_lab = 'expr'
hue = 'type'

ax = sns.boxplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('log CPM', fontsize=12)
ax.set_title('ATAC peak counts inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure2/atac_diff_expr_in_pmds.csv', sep='\t', index=None)

### how about all peaks?

In [ ]:
clust_list = []

tmp_peaks = peaks.copy()
tmp_peaks = tmp_peaks[~tmp_peaks.index.duplicated()]
tmp_peaks_bt = BedTool.from_dataframe(tmp_peaks)
tmp_peaks_pmds = tmp_peaks_bt.intersect(pmds_bt, wa=True).to_dataframe()
tmp_peaks_pmds.columns = tmp_peaks.columns
tmp_peaks_pmds.index = [f'{x}_{y}' for x,y in zip(tmp_peaks_pmds['chrom'], tmp_peaks_pmds['start'])]

for s in tmp_order:

    ###
    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]
    mask = tmp_pseudo.columns.isin(tmp_peaks.index)
    tmp_pseudo = tmp_pseudo.loc[:,mask]
    
    ###
    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(tmp_peaks_pmds.index)
    merged_pmds = merged_expr[mask].copy()
    mask = ~merged_expr.index.isin(tmp_peaks_pmds.index)
    merged_comp = merged_expr[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']
    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'cluster'
y_lab = 'expr'
hue = 'type'

ax = sns.boxplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('log CPM', fontsize=12)
ax.set_title('ATAC peak counts inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure2/atac_expr_in_pmds.csv', sep='\t', index=None)

### check all donors

In [ ]:
log_pseudo = np.log(norm_pseudo + 1)
log_pseudo.shape

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_peaks = peaks[peaks['coarse_v7'] == s]
    tmp_peaks = tmp_peaks[~tmp_peaks.index.duplicated()]
    tmp_peaks_bt = BedTool.from_dataframe(tmp_peaks)
    tmp_peaks_pmds = tmp_peaks_bt.intersect(pmds_bt, wa=True).to_dataframe()
    tmp_peaks_pmds.columns = tmp_peaks.columns
    tmp_peaks_pmds.index = [f'{x}_{y}' for x,y in zip(tmp_peaks_pmds['chrom'], tmp_peaks_pmds['start'])]
    
    ###
    mask = [x.split('_')[0] == s for x in log_pseudo.index]
    tmp_pseudo = log_pseudo[mask].T
    mask = tmp_pseudo.index.isin(tmp_peaks.index)
    tmp_pseudo = tmp_pseudo[mask]

    ####
    mask = tmp_pseudo.index.isin(tmp_peaks_pmds.index)
    merged_pmds = tmp_pseudo[mask].copy()
    mask = tmp_pseudo.index.isin(tmp_peaks_comp.index)
    merged_comp = tmp_pseudo[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    concat = pd.concat([merged_pmds, merged_comp])
    to_plot = concat.melt(id_vars=concat.columns[-1],value_vars=concat.columns[:-1])
    to_plot['cluster'] = [x.split('_')[0] for x in to_plot['cluster_donor']]
    to_plot['donor'] = [x.split('_')[1] for x in to_plot['cluster_donor']]
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
s = 'IPS'
pmds_plot[pmds_plot['cluster'] == s]

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 8))

x_lab = 'donor'
y_lab = 'value'
hue = 'type'

for i,ax in enumerate(axes.flatten()):
    s = tmp_order[i]
    to_plot = pmds_plot[pmds_plot['cluster'] == s]
    sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue,showfliers=False, ax=ax)
    ax.set_xlabel('', fontsize=10)
    ax.set_ylabel('log CPM', fontsize=10)
    ax.set_title(f'{s}', fontsize=14)
    ax.tick_params(axis='x', labelsize=12, labelrotation=0)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    
plt.subplots_adjust(hspace=0.6, wspace=0.4)

In [ ]:
!date